# CWRU fault classifier and per-motor anomaly detector

This notebook compares four classes from drive-end vibration: normal, inner-race, ball, and centered outer-race faults. It evaluates leave-one-load-out transfer and a separate held-out fault-size experiment. Every segment from a recording inherits that recording's split.

The absolute-feature classifier is the Decision 003 reference. A second classifier replaces RMS and absolute envelope energies with kurtosis, crest factor, fault-band energy ratios, and envelope peak-to-neighbor ratios. A healthy-baseline detector is also evaluated: it learns only from healthy recordings available in each training split and flags feature deviations. The notebook handles CWRU loading, plots, and evaluation; reusable features and baseline scoring live in `ml/src/motorsense_ml`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from motorsense_ml.dataset import segment_signal
from motorsense_ml.features import (
    crest_factor,
    envelope_band_energies,
    envelope_scale_invariant_features,
    kurtosis,
    rms,
)
from motorsense_ml.healthy_baseline import (
    HealthyBaselineDetector,
    leave_one_recording_out_scores,
)
from motorsense_ml.resampling import resample_signal

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "ml" / "data").is_dir())
DATA_DIR = ROOT / "ml" / "data"
TARGET_FS = 12_000
SOURCE_FS_BY_CLASS = {"normal": 48_000, "inner race": 12_000, "ball": 12_000, "outer race": 12_000}
RPM_BY_LOAD = {0: 1797, 1: 1772, 2: 1750, 3: 1730}  # CWRU table approximations
SEGMENT_SAMPLES = TARGET_FS // 2  # half-second windows; 2 Hz envelope FFT bins
SEGMENTS_PER_RECORDING = 10
FAULT_IDS = {
    "inner race": {
        0.007: (105, 106, 107, 108),
        0.014: (169, 170, 171, 172),
        0.021: (209, 210, 211, 212),
    },
    "ball": {0.007: (118, 119, 120, 121), 0.014: (185, 186, 187, 188), 0.021: (222, 223, 224, 225)},
    "outer race": {
        0.007: (130, 131, 132, 133),
        0.014: (197, 198, 199, 200),
        0.021: (234, 235, 236, 237),
    },
}

## Check the normal recordings' sample-rate assumption

The CWRU pages document the normal files by load and say the drive-end collection includes 12 kHz and 48 kHz data, but they do not give a per-file normal-baseline rate. Decision 001 verified `97.mat` at 48 kHz by comparing its shaft harmonics with the known 12 kHz `105.mat`. We treat `98.mat`–`100.mat` as the same 48 kHz normal-baseline collection; their sample counts are about 484k, consistent with a roughly 10 s 48 kHz record. This is a collection-level assumption, not explicit per-file metadata. The code reports duration under each candidate so the assumption remains visible.


In [ ]:
def drive_end_signal(file_id):
    mat = loadmat(DATA_DIR / f"{file_id}.mat")
    key = f"X{file_id:03d}_DE_time"
    if key not in mat:
        raise KeyError(f"{key} missing from {file_id}.mat")
    return np.asarray(mat[key], dtype=float).reshape(-1)


for load_hp, file_id in enumerate((97, 98, 99, 100)):
    signal = drive_end_signal(file_id)
    print(
        f"{file_id}.mat ({load_hp} HP): {signal.size:,} samples; "
        f"{signal.size / 48_000:.2f} s at 48 kHz or {signal.size / 12_000:.2f} s at 12 kHz"
    )

The sample-count check above does not establish the rate by itself: duration depends on the assumed rate. We use 48 kHz for the normal-baseline collection because the CWRU record set is the 48 kHz normal baseline; the official page does not provide per-file rate metadata. The existing `97.mat` harmonic comparison is documented in Decision 001. We resample the normal signals to 12 kHz so their feature windows match the 12 kHz fault recordings.


## Build fixed-length feature rows

Each row represents one non-overlapping 0.5 s segment. We keep ten evenly spaced segments from each recording so one long file does not dominate. The split is assigned from `load_hp` after features are extracted, and the test set is the entire 3 HP recording for each class/size. No recording contributes segments to both sides.

Envelope energies sum squared envelope-spectrum magnitudes around the first three harmonics of BPFI, BPFO, BSF, and FTF. Those orders are based on the CWRU drive-end 6205 bearing. The feature helper returns them in that fixed order on a log scale.


In [ ]:
recording_specs = []
for file_id, load_hp in zip((97, 98, 99, 100), range(4), strict=True):
    recording_specs.append(
        {
            "file_id": file_id,
            "label": "normal",
            "load_hp": load_hp,
            "fault_size_in": 0.0,
            "outer_position": None,
        }
    )
for label, sizes in FAULT_IDS.items():
    for size, file_ids in sizes.items():
        for load_hp, file_id in enumerate(file_ids):
            recording_specs.append(
                {
                    "file_id": file_id,
                    "label": label,
                    "load_hp": load_hp,
                    "fault_size_in": size,
                    "outer_position": "centered @ 6" if label == "outer race" else None,
                }
            )

feature_names = [
    "rms",
    "kurtosis",
    "crest_factor",
    "log_bpfi_energy",
    "log_bpfo_energy",
    "log_bsf_energy",
    "log_ftf_energy",
]
scale_feature_names = [
    "kurtosis",
    "crest_factor",
    "bpfi_energy_ratio",
    "bpfo_energy_ratio",
    "bsf_energy_ratio",
    "ftf_energy_ratio",
    "bpfi_peak_neighbor_db",
    "bpfo_peak_neighbor_db",
    "bsf_peak_neighbor_db",
    "ftf_peak_neighbor_db",
]
feature_rows = []
recording_rows = []
for spec in recording_specs:
    file_id = spec["file_id"]
    signal = drive_end_signal(file_id)
    mat = loadmat(DATA_DIR / f"{file_id}.mat")
    rpm_key = f"X{file_id:03d}RPM"
    rpm = (
        float(np.asarray(mat[rpm_key]).squeeze())
        if rpm_key in mat
        else RPM_BY_LOAD[spec["load_hp"]]
    )
    source_fs = SOURCE_FS_BY_CLASS[spec["label"]]
    if source_fs != TARGET_FS:
        signal = resample_signal(signal, source_fs, TARGET_FS)
    segments = segment_signal(signal, SEGMENT_SAMPLES, max_segments=SEGMENTS_PER_RECORDING)
    recording_id = f"{file_id:03d}"
    recording_rows.append(
        {
            **spec,
            "recording_id": recording_id,
            "rpm_used": rpm,
            "source_fs": source_fs,
            "segment_count": len(segments),
        }
    )
    for segment_index, segment in enumerate(segments):
        env = envelope_band_energies(segment, TARGET_FS, rpm / 60)
        scale_env = envelope_scale_invariant_features(segment, TARGET_FS, rpm / 60)
        segment_kurtosis = kurtosis(segment)
        segment_crest = crest_factor(segment)
        values = [rms(segment), segment_kurtosis, segment_crest, *env]
        scale_values = [segment_kurtosis, segment_crest, *scale_env]
        feature_rows.append(
            {
                "recording_id": recording_id,
                "segment_index": segment_index,
                "label": spec["label"],
                "load_hp": spec["load_hp"],
                "fault_size_in": spec["fault_size_in"],
                **dict(zip(feature_names, values, strict=True)),
                **dict(zip(scale_feature_names, scale_values, strict=True)),
            }
        )

print(f"{len(recording_rows)} recordings, {len(feature_rows)} segments")
for load_hp in range(4):
    counts = {
        label: sum(row["load_hp"] == load_hp and row["label"] == label for row in recording_rows)
        for label in ("normal", "inner race", "ball", "outer race")
    }
    print(f"{load_hp} HP: {counts}")
print("Example feature row:", feature_rows[0])

## Leave-one-load-out cross-validation

Each outer fold holds out all recordings at one motor load. The inner model-selection folds group by recording and use only the other three loads. This reports both window-level scores (400 half-second windows total) and recording-level scores (40 recordings total); the latter aggregates each recording's window probabilities before scoring.


In [ ]:
from sklearn.metrics import accuracy_score, recall_score

labels = ["normal", "inner race", "ball", "outer race"]


def make_models():
    return {
        "dummy prior": DummyClassifier(strategy="prior"),
        "logistic regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=2_000, random_state=42),
        ),
        "random forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    }


def arrays_for(rows, names=feature_names):
    matrix = np.asarray([[row[name] for name in names] for row in rows])
    y = np.asarray([row["label"] for row in rows])
    groups = np.asarray([row["recording_id"] for row in rows])
    return matrix, y, groups


def select_model(train_rows, names=feature_names):
    matrix, y, groups = arrays_for(train_rows, names)
    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    scores = {}
    for name, estimator in make_models().items():
        values = cross_val_score(
            estimator, matrix, y, groups=groups, cv=splitter, scoring="balanced_accuracy"
        )
        scores[name] = float(values.mean())
    selected = max((name for name in scores if name != "dummy prior"), key=scores.get)
    model = make_models()[selected].fit(matrix, y)
    return selected, model, scores[selected]


def recording_predictions(rows, probabilities, classes):
    grouped = {}
    for index, row in enumerate(rows):
        grouped.setdefault(row["recording_id"], []).append(index)
    result = []
    for recording_id, indices in grouped.items():
        first = rows[indices[0]]
        mean_probability = probabilities[indices].mean(axis=0)
        result.append(
            {
                "recording_id": recording_id,
                "actual": first["label"],
                "predicted": str(classes[int(np.argmax(mean_probability))]),
                "load_hp": first["load_hp"],
                "fault_size_in": first["fault_size_in"],
            }
        )
    return result


def evaluate_anomaly_detector(train_rows, test_rows):
    # Even-indexed healthy windows fit the baseline and calibrate its threshold.
    # Odd-indexed windows from those same recordings are reserved for a false-alarm check.
    healthy_rows = [row for row in train_rows if row["label"] == "normal"]
    baseline_rows = [row for row in healthy_rows if row["segment_index"] % 2 == 0]
    unused_healthy_rows = [row for row in healthy_rows if row["segment_index"] % 2 == 1]
    healthy_x, _, healthy_groups = arrays_for(baseline_rows, scale_feature_names)
    calibration_scores = leave_one_recording_out_scores(healthy_x, healthy_groups)
    threshold = float(np.quantile(calibration_scores, 0.95))
    detector = HealthyBaselineDetector().fit(healthy_x)

    test_x, _, _ = arrays_for(test_rows, scale_feature_names)
    scores = detector.decision_function(test_x)
    y_window = np.asarray([row["label"] != "normal" for row in test_rows], dtype=int)
    pred_window = (scores > threshold).astype(int)
    grouped = {}
    for index, row in enumerate(test_rows):
        grouped.setdefault(row["recording_id"], []).append(index)
    y_record, pred_record = [], []
    for indices in grouped.values():
        first = test_rows[indices[0]]
        y_record.append(int(first["label"] != "normal"))
        pred_record.append(int(np.median(scores[indices]) > threshold))

    heldout_healthy_rows = [row for row in test_rows if row["label"] == "normal"]
    healthy_eval_rows = [*heldout_healthy_rows, *unused_healthy_rows]
    healthy_eval_x, _, _ = arrays_for(healthy_eval_rows, scale_feature_names)
    healthy_eval_scores = detector.decision_function(healthy_eval_x)
    heldout_count = len(heldout_healthy_rows)
    return {
        "threshold": threshold,
        "window_actual": y_window,
        "window_predicted": pred_window,
        "recording_actual": np.asarray(y_record),
        "recording_predicted": np.asarray(pred_record),
        "heldout_healthy_false_alarms": int(
            np.sum(healthy_eval_scores[:heldout_count] > threshold)
        ),
        "heldout_healthy_count": heldout_count,
        "heldout_healthy_window_ids": [
            (row["recording_id"], row["segment_index"]) for row in heldout_healthy_rows
        ],
        "heldout_healthy_alarm_ids": [
            (row["recording_id"], row["segment_index"])
            for row, score in zip(
                heldout_healthy_rows, healthy_eval_scores[:heldout_count], strict=True
            )
            if score > threshold
        ],
        "unused_healthy_false_alarms": int(np.sum(healthy_eval_scores[heldout_count:] > threshold)),
        "unused_healthy_count": len(unused_healthy_rows),
        "unused_healthy_window_ids": [
            (row["recording_id"], row["segment_index"]) for row in unused_healthy_rows
        ],
        "unused_healthy_alarm_ids": [
            (row["recording_id"], row["segment_index"])
            for row, score in zip(
                unused_healthy_rows, healthy_eval_scores[heldout_count:], strict=True
            )
            if score > threshold
        ],
    }


load_summaries = []
scale_load_summaries = []
anomaly_summaries = []
window_actual, window_predicted = [], []
recording_actual, recording_predicted = [], []
scale_window_actual, scale_window_predicted = [], []
scale_recording_actual, scale_recording_predicted = [], []
anomaly_window_actual, anomaly_window_predicted = [], []
anomaly_recording_actual, anomaly_recording_predicted = [], []
heldout_healthy_false_alarms = 0
heldout_healthy_window_count = 0
unused_healthy_false_alarms = 0
unused_healthy_window_count = 0
heldout_healthy_window_ids = set()
heldout_healthy_alarm_ids = set()
unused_healthy_window_ids = set()
unused_healthy_alarm_ids = set()
rf_importances = []
for heldout_load in range(4):
    fit_rows = [row for row in feature_rows if row["load_hp"] != heldout_load]
    heldout_rows = [row for row in feature_rows if row["load_hp"] == heldout_load]
    fit_ids = {row["recording_id"] for row in fit_rows}
    heldout_ids = {row["recording_id"] for row in heldout_rows}
    assert fit_ids.isdisjoint(heldout_ids)

    X_fit, y_fit, _ = arrays_for(fit_rows)
    X_heldout, y_heldout, _ = arrays_for(heldout_rows)
    model_name, model, inner_score = select_model(fit_rows)
    y_window_pred = model.predict(X_heldout)
    probabilities = model.predict_proba(X_heldout)
    rec_rows = recording_predictions(heldout_rows, probabilities, model.classes_)
    rec_actual = [row["actual"] for row in rec_rows]
    rec_predicted = [row["predicted"] for row in rec_rows]
    load_summaries.append(
        {
            "heldout_load": heldout_load,
            "selected_model": model_name,
            "inner_cv": inner_score,
            "window_accuracy": accuracy_score(y_heldout, y_window_pred),
            "window_bacc": balanced_accuracy_score(y_heldout, y_window_pred),
            "recording_accuracy": accuracy_score(rec_actual, rec_predicted),
            "recording_bacc": balanced_accuracy_score(rec_actual, rec_predicted),
        }
    )
    window_actual.extend(y_heldout)
    window_predicted.extend(y_window_pred)
    recording_actual.extend(rec_actual)
    recording_predicted.extend(rec_predicted)

    scale_model_name, scale_model, scale_inner_score = select_model(fit_rows, scale_feature_names)
    X_scale_test, y_scale_test, _ = arrays_for(heldout_rows, scale_feature_names)
    scale_window_pred = scale_model.predict(X_scale_test)
    scale_probabilities = scale_model.predict_proba(X_scale_test)
    scale_rec_rows = recording_predictions(heldout_rows, scale_probabilities, scale_model.classes_)
    scale_rec_actual = [row["actual"] for row in scale_rec_rows]
    scale_rec_predicted = [row["predicted"] for row in scale_rec_rows]
    scale_load_summaries.append(
        {
            "heldout_load": heldout_load,
            "selected_model": scale_model_name,
            "inner_cv": scale_inner_score,
            "window_accuracy": accuracy_score(y_scale_test, scale_window_pred),
            "window_bacc": balanced_accuracy_score(y_scale_test, scale_window_pred),
            "recording_accuracy": accuracy_score(scale_rec_actual, scale_rec_predicted),
            "recording_bacc": balanced_accuracy_score(scale_rec_actual, scale_rec_predicted),
        }
    )
    scale_window_actual.extend(y_scale_test)
    scale_window_predicted.extend(scale_window_pred)
    scale_recording_actual.extend(scale_rec_actual)
    scale_recording_predicted.extend(scale_rec_predicted)

    anomaly = evaluate_anomaly_detector(fit_rows, heldout_rows)
    anomaly_summaries.append(
        {
            "heldout_load": heldout_load,
            "threshold": anomaly["threshold"],
            "window_bacc": balanced_accuracy_score(
                anomaly["window_actual"], anomaly["window_predicted"]
            ),
            "recording_bacc": balanced_accuracy_score(
                anomaly["recording_actual"], anomaly["recording_predicted"]
            ),
        }
    )
    anomaly_window_actual.extend(anomaly["window_actual"])
    anomaly_window_predicted.extend(anomaly["window_predicted"])
    anomaly_recording_actual.extend(anomaly["recording_actual"])
    anomaly_recording_predicted.extend(anomaly["recording_predicted"])
    heldout_healthy_false_alarms += anomaly["heldout_healthy_false_alarms"]
    heldout_healthy_window_count += anomaly["heldout_healthy_count"]
    unused_healthy_false_alarms += anomaly["unused_healthy_false_alarms"]
    unused_healthy_window_count += anomaly["unused_healthy_count"]
    heldout_healthy_window_ids.update(anomaly["heldout_healthy_window_ids"])
    heldout_healthy_alarm_ids.update(anomaly["heldout_healthy_alarm_ids"])
    unused_healthy_window_ids.update(anomaly["unused_healthy_window_ids"])
    unused_healthy_alarm_ids.update(anomaly["unused_healthy_alarm_ids"])

    diagnostic_rf = make_models()["random forest"].fit(X_fit, y_fit)
    rf_importances.append(diagnostic_rf.feature_importances_)

for baseline, scaled in zip(load_summaries, scale_load_summaries, strict=True):
    print(
        f"{baseline['heldout_load']} HP | absolute {baseline['window_bacc']:.3f}/"
        f"{baseline['recording_bacc']:.3f} | scale-invariant "
        f"{scaled['window_bacc']:.3f}/{scaled['recording_bacc']:.3f}"
    )
print(
    "All LOLO folds (window/recording balanced accuracy) | absolute "
    f"{balanced_accuracy_score(window_actual, window_predicted):.3f}/"
    f"{balanced_accuracy_score(recording_actual, recording_predicted):.3f} | "
    "scale-invariant "
    f"{balanced_accuracy_score(scale_window_actual, scale_window_predicted):.3f}/"
    f"{balanced_accuracy_score(scale_recording_actual, scale_recording_predicted):.3f}"
)
print(
    "Healthy-baseline anomaly balanced accuracy (window/recording): "
    f"{balanced_accuracy_score(anomaly_window_actual, anomaly_window_predicted):.3f}/"
    f"{balanced_accuracy_score(anomaly_recording_actual, anomaly_recording_predicted):.3f}"
)
print("Anomaly window confusion matrix [healthy, fault]:")
print(confusion_matrix(anomaly_window_actual, anomaly_window_predicted, labels=[0, 1]))
print("Anomaly recording confusion matrix [healthy, fault]:")
print(confusion_matrix(anomaly_recording_actual, anomaly_recording_predicted, labels=[0, 1]))
print(
    "Held-out-load healthy windows false alarms: "
    f"{heldout_healthy_false_alarms}/{heldout_healthy_window_count} "
    f"({heldout_healthy_false_alarms / heldout_healthy_window_count:.1%})"
)
print(
    "Unused healthy windows from other-load recordings false alarms: "
    f"{unused_healthy_false_alarms}/{unused_healthy_window_count} "
    f"({unused_healthy_false_alarms / unused_healthy_window_count:.1%}; "
    "each recording's reserved windows are evaluated in three fold contexts)"
)
unique_healthy_window_ids = heldout_healthy_window_ids | unused_healthy_window_ids
unique_healthy_alarm_ids = heldout_healthy_alarm_ids | unused_healthy_alarm_ids
print(
    "Unique healthy windows flagged at least once: "
    f"{len(unique_healthy_alarm_ids)}/{len(unique_healthy_window_ids)} "
    f"({len(unique_healthy_alarm_ids) / len(unique_healthy_window_ids):.1%})"
)
print(
    f"Reserved segments: {len(unused_healthy_alarm_ids)}/{len(unused_healthy_window_ids)} "
    "unique windows flagged in any of their three fold contexts"
)

importance_array = np.asarray(rf_importances)
importance_mean = importance_array.mean(axis=0)
importance_std = importance_array.std(axis=0)
for index in np.argsort(importance_mean)[::-1]:
    print(
        f"RF impurity importance {feature_names[index]}: "
        f"{importance_mean[index]:.3f} +/- {importance_std[index]:.3f}"
    )

## Fault-size holdout

Train on normal recordings plus 0.007 and 0.014 inch faults. Test only on 0.021 inch fault recordings across all four loads. This asks whether the model can identify the fault type at a size it never saw. Normal recordings have no fault size, so this holdout measures classification among the three fault types; it does not measure normal-versus-fault detection.


In [ ]:
fault_size = 0.021
size_train = [
    row for row in feature_rows if row["label"] == "normal" or row["fault_size_in"] != fault_size
]
size_test = [
    row for row in feature_rows if row["label"] != "normal" and row["fault_size_in"] == fault_size
]
size_train_ids = {row["recording_id"] for row in size_train}
size_test_ids = {row["recording_id"] for row in size_test}
assert size_train_ids.isdisjoint(size_test_ids)
assert {row["fault_size_in"] for row in size_test} == {fault_size}
assert {row["label"] for row in size_test} == {"inner race", "ball", "outer race"}

# Both classifiers use precisely the same training and held-out recordings.
size_model_name, size_model, size_inner_score = select_model(size_train)
X_size_test, y_size_test, _ = arrays_for(size_test)
y_size_window_pred = size_model.predict(X_size_test)
size_probabilities = size_model.predict_proba(X_size_test)
size_record_rows = recording_predictions(size_test, size_probabilities, size_model.classes_)
y_size_record_actual = [row["actual"] for row in size_record_rows]
y_size_record_pred = [row["predicted"] for row in size_record_rows]

size_scale_model_name, size_scale_model, size_scale_inner_score = select_model(
    size_train, scale_feature_names
)
X_size_scale_test, y_size_scale_test, _ = arrays_for(size_test, scale_feature_names)
y_size_scale_window_pred = size_scale_model.predict(X_size_scale_test)
size_scale_probabilities = size_scale_model.predict_proba(X_size_scale_test)
size_scale_record_rows = recording_predictions(
    size_test, size_scale_probabilities, size_scale_model.classes_
)
y_size_scale_record_actual = [row["actual"] for row in size_scale_record_rows]
y_size_scale_record_pred = [row["predicted"] for row in size_scale_record_rows]

size_anomaly = evaluate_anomaly_detector(size_train, size_test)
print(f"Training recordings/windows: {len(size_train_ids)}/{len(size_train)}")
print(f"Held-out recordings/windows: {len(size_test_ids)}/{len(size_test)}")
for name, inner, y_true, y_window, y_record, y_record_pred in (
    (
        "Absolute classifier",
        size_inner_score,
        y_size_test,
        y_size_window_pred,
        y_size_record_actual,
        y_size_record_pred,
    ),
    (
        "Scale-invariant classifier",
        size_scale_inner_score,
        y_size_scale_test,
        y_size_scale_window_pred,
        y_size_scale_record_actual,
        y_size_scale_record_pred,
    ),
):
    print(
        f"{name} ({size_model_name if name == 'Absolute classifier' else size_scale_model_name}; "
        f"inner grouped CV {inner:.3f}) | window accuracy/balanced accuracy "
        f"{accuracy_score(y_true, y_window):.3f}/{balanced_accuracy_score(y_true, y_window):.3f} | "
        f"recording accuracy/balanced accuracy "
        f"{accuracy_score(y_record, y_record_pred):.3f}/"
        f"{balanced_accuracy_score(y_record, y_record_pred):.3f}"
    )
print("Absolute classifier fault-size recording confusion matrix:")
print(confusion_matrix(y_size_record_actual, y_size_record_pred, labels=labels))
print("Scale-invariant classifier fault-size recording confusion matrix:")
print(confusion_matrix(y_size_scale_record_actual, y_size_scale_record_pred, labels=labels))
print(
    "Per-motor baseline, fault-size sensitivity (window/recording; no normal negatives "
    "in this test set): "
    f"{recall_score(size_anomaly['window_actual'], size_anomaly['window_predicted']):.3f}/"
    f"{recall_score(size_anomaly['recording_actual'], size_anomaly['recording_predicted']):.3f}"
)
print("Fault-size anomaly recording outcomes [actual, predicted]:")
print(list(zip(size_anomaly["recording_actual"], size_anomaly["recording_predicted"], strict=True)))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
matrices = [
    (confusion_matrix(window_actual, window_predicted, labels=labels), "LOLO absolute windows"),
    (
        confusion_matrix(scale_window_actual, scale_window_predicted, labels=labels),
        "LOLO scale-invariant windows",
    ),
    (
        confusion_matrix(recording_actual, recording_predicted, labels=labels),
        "LOLO absolute recordings",
    ),
    (
        confusion_matrix(scale_recording_actual, scale_recording_predicted, labels=labels),
        "LOLO scale-invariant recordings",
    ),
    (
        confusion_matrix(y_size_record_actual, y_size_record_pred, labels=labels),
        "0.021 in absolute classifier",
    ),
    (
        confusion_matrix(y_size_scale_record_actual, y_size_scale_record_pred, labels=labels),
        "0.021 in scale-invariant",
    ),
]
for axis, (matrix, title) in zip(axes.flat, matrices, strict=True):
    ConfusionMatrixDisplay(matrix, display_labels=labels).plot(
        ax=axis, cmap="Blues", values_format="d", colorbar=False
    )
    axis.set_title(title)
    axis.tick_params(axis="x", labelrotation=35)
plt.tight_layout()
plt.show()

## Interpretation limits

The absolute-feature classifier repeats the Decision 003 protocol, including nested grouped model selection; the scale-invariant classifier is selected independently within the same training recordings and evaluated on the same held-out load and fault-size recordings. Window metrics help show segment behavior, but windows from one recording are correlated, so recording-level metrics are the stronger independent-unit summary.

The anomaly detector uses only even-indexed healthy training windows to fit its feature center/scales and to calibrate a 95th-percentile threshold from leave-one-recording-out scores. Odd-indexed windows from those same normal recordings are never used by the detector; they provide the reserved-window false-alarm estimate. Healthy windows from the held-out load provide a second, recording-level holdout estimate. On load holdouts, both healthy and faulty recordings measure balanced accuracy. The fault-size test contains only faults, so it reports sensitivity only; it cannot estimate false alarms or specificity.

Feature importance from Decision 003 is a model diagnostic, not evidence that a feature causes classification. All recordings come from one laboratory dataset and one motor setup; these comparisons do not establish field performance or behavior on a newly commissioned motor. A deployed per-motor baseline would need a deliberate healthy commissioning period and drift handling.
